# Risk Actions: Pre-LLM vs Post-LLM Explanations

This notebook is a learning notebook. The goal is to make the LLM layer visible instead of hidden inside the dashboard.

We will compare:

- **Pre-LLM explanation**: deterministic text created from the `HoldingActionRecommendation` object.
- **Post-LLM explanation**: plain-English rewrite generated by Ollama from the same deterministic evidence.

Important rule for this project:

**Math decides. Evidence supports. LLM explains. UI earns trust.**

## Agentic Lifecycle For This Notebook

Think of the LLM explanation layer as an agentic step with guardrails:

1. **Observe**: Load the saved portfolio diagnosis and action recommendations.
2. **Diagnose**: Identify the holdings that already passed the rule-based sell/trim gate.
3. **Plan**: Build a compact evidence payload for each holding.
4. **Act**: Ask Ollama to rewrite the evidence in plain English.
5. **Verify**: Compare deterministic text vs LLM text side by side.
6. **Explain**: Confirm that the LLM did not change the action, amount, ranking, or confidence.

The LLM is not a portfolio manager here. It is a communication layer.

In [8]:
from pathlib import Path
import json
import sys

import pandas as pd

# Make the notebook robust whether you open it from repo root or data_pipeline.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "data_pipeline" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DIAGNOSIS_DIR = PROJECT_ROOT / "data" / "processed" / "diagnosis"
DIAGNOSIS_DIR

PosixPath('/Users/sagnikrana/Documents/GitHub/portfolio-analyzer/data/processed/diagnosis')

In [9]:
from portfolio_analyzer.diagnosis import portfolio_risk_diagnosis_from_saved_artifacts
from portfolio_analyzer.app import (
    _clean_llm_explanation,
    _extract_json_object,
    _risk_action_deterministic_explanation,
    _risk_action_llm_payload,
    call_ollama,
    money_text,
    percent_display,
    preferred_risk_action_llm_model,
)

from IPython.display import HTML, display

diagnosis = portfolio_risk_diagnosis_from_saved_artifacts(DIAGNOSIS_DIR)
actionable = [item for item in diagnosis.holding_action_recommendations if item.is_actionable]

print(f"Loaded diagnosis run: {diagnosis.run_id}")
print(f"Actionable Risk Actions: {len(actionable)}")
print(f"App preferred Ollama model: {preferred_risk_action_llm_model()}")



Loaded diagnosis run: 20260505_071554
Actionable Risk Actions: 10
App preferred Ollama model: llama3.3:latest


## 1. Pre-LLM Explanations

This table is the pre-LLM baseline. It is created directly from the deterministic action object.

If the app had no LLM available, this is the fallback explanation style users would still see.

In [10]:
def explanation_to_row(item, explanation, prefix):
    return {
        "Ticker": item.ticker,
        "Action": item.recommendation_label,
        "Value to sell": money_text(item.value_to_sell),
        "Reduction %": percent_display(item.position_reduction_pct),
        "Confidence": item.confidence_band,
        "Source": explanation.get("_source", "Unknown"),
        f"{prefix} headline": explanation.get("headline", ""),
        f"{prefix} summary": explanation.get("plain_english_summary", ""),
        f"{prefix} why now": explanation.get("why_now", ""),
        f"{prefix} context": explanation.get("external_context", ""),
        f"{prefix} improves": explanation.get("what_improves", ""),
        f"{prefix} watchout": explanation.get("watchout", ""),
    }

pre_llm_rows = [
    explanation_to_row(item, _risk_action_deterministic_explanation(item), "Pre-LLM")
    for item in actionable
]

pre_llm_df = pd.DataFrame(pre_llm_rows)
pre_llm_df

,Ticker,Action,Value to sell,Reduction %,Confidence,Source,Pre-LLM headline,Pre-LLM summary,Pre-LLM why now,Pre-LLM context,Pre-LLM improves,Pre-LLM watchout
0,NOW,Trim 35%,$79.40,35.00%,Medium,Rule-based fallback,Trim 35% NOW: the holding has lagged the marke...,NOW is currently a **trim 35%** candidate. NOW...,NOW has lagged the trade-matched S&P 500 by 74...,"No extra company-report, news, or macro signal...","If you make this move, the portfolio should be...","This is a portfolio-risk action, not a predict..."
1,MSFT,Trim 35%,"$3,100.11",35.00%,High,Rule-based fallback,Trim 35% MSFT: the holding has lagged the mark...,MSFT is currently a **trim 35%** candidate. MS...,MSFT has lagged the trade-matched S&P 500 by 3...,**Recent company reports or news added caution...,"If you make this move, the portfolio should be...","This is a portfolio-risk action, not a predict..."
2,SOFI,Trim 25%,$169.86,25.00%,Medium,Rule-based fallback,Trim 25% SOFI: the holding has lagged the mark...,SOFI is currently a **trim 25%** candidate. SO...,SOFI has lagged the trade-matched S&P 500 by 4...,"No extra company-report, news, or macro signal...","If you make this move, the portfolio should be...","Some longer-horizon strength is still present,..."
3,BRK.A,Trim 25%,$43.92,25.00%,Medium,Rule-based fallback,Trim 25% BRK.A: the holding has lagged the mar...,BRK.A is currently a **trim 25%** candidate. B...,BRK.A has lagged the trade-matched S&P 500 by ...,"No extra company-report, news, or macro signal...","If you make this move, the portfolio should be...","This is a portfolio-risk action, not a predict..."
4,TCEHY,Trim 25%,$105.73,25.00%,Medium,Rule-based fallback,Trim 25% TCEHY: the holding has lagged the mar...,TCEHY is currently a **trim 25%** candidate. T...,TCEHY has lagged the trade-matched S&P 500 by ...,"No extra company-report, news, or macro signal...","If you make this move, the portfolio should be...","This is a portfolio-risk action, not a predict..."
5,UBER,Trim 25%,$209.10,25.00%,Medium,Rule-based fallback,Trim 25% UBER: the holding has lagged the mark...,UBER is currently a **trim 25%** candidate. UB...,UBER has lagged the trade-matched S&P 500 by 3...,"No extra company-report, news, or macro signal...","If you make this move, the portfolio should be...","Some longer-horizon strength is still present,..."
6,BRK.B,Trim 25%,$468.52,25.00%,Medium,Rule-based fallback,Trim 25% BRK.B: the holding has lagged the mar...,BRK.B is currently a **trim 25%** candidate. B...,BRK.B has lagged the trade-matched S&P 500 by ...,"No extra company-report, news, or macro signal...","If you make this move, the portfolio should be...","This is a portfolio-risk action, not a predict..."
7,BABA,Trim 25%,$121.70,25.00%,Medium,Rule-based fallback,Trim 25% BABA: the holding has lagged the mark...,BABA is currently a **trim 25%** candidate. BA...,BABA has lagged the trade-matched S&P 500 by 1...,"No extra company-report, news, or macro signal...","If you make this move, the portfolio should be...","This is a portfolio-risk action, not a predict..."
8,MMYT,Trim 15%,$13.26,15.00%,Medium,Rule-based fallback,Trim 15% MMYT: the holding has lagged the mark...,MMYT is currently a **trim 15%** candidate. MM...,MMYT has lagged the trade-matched S&P 500 by 7...,**Higher interest rates are still a headwind**...,"If you make this move, the portfolio should be...",Strong trailing outperformance in multiple lon...
9,META,Trim 10%,$790.78,10.00%,High,Rule-based fallback,Trim 10% META: the holding has lagged the mark...,META is currently a **trim 10%** candidate. ME...,META has lagged the trade-matched S&P 500 by 2...,**Recent company reports or news added caution...,"If you make this move, the portfolio should be...",Strong trailing outperformance in multiple lon...


## 2. Evidence Payload Sent To The LLM

This is the most important trust-building cell.

The LLM is not allowed to look around freely. It only receives this compact JSON payload, which includes:

- the already-decided action and amount
- recent performance versus the S&P 500
- portfolio-risk evidence
- external context from company reports, news, fundamentals, and macro signals
- existing deterministic explanations and guardrails

In [15]:
if actionable:
    first_action = actionable[0]
    payload = _risk_action_llm_payload(diagnosis, first_action)
    print(f"Showing LLM payload for: {first_action.ticker}")
    print(json.dumps(payload, indent=2)[:8000])
else:
    print("No actionable holdings were available in the latest diagnosis.")

Showing LLM payload for: NOW
{
  "ticker": "NOW",
  "sector": "Technology",
  "precomputed_action": {
    "label": "Trim 35%",
    "shares_to_sell": 0.8633,
    "value_to_sell": 79.4,
    "position_reduction_pct": 0.35,
    "current_weight": 0.0015,
    "target_weight_after_action": 0.001,
    "confidence_band": "Medium"
  },
  "performance_evidence": {
    "since_buy_window": "Since weighted-average buy date (2024-11-08) through 2026-04-01",
    "holding_return": -0.5463,
    "sp500_return_same_window": 0.201,
    "relative_vs_sp500_same_window": -0.7473,
    "relative_1y": -0.8042131007738742,
    "relative_3y": -0.7048991941645284,
    "relative_5y": -0.7775332386896249
  },
  "portfolio_risk_evidence": {
    "primary_concern": null,
    "diagnosis_pressure_score": 0.0,
    "projected_weight_reduction": 0.0005,
    "projected_variance_reduction": 0.0,
    "projected_lag_reduction": 0.0004
  },
  "existing_explanation": {
    "summary": "NOW is currently a **trim 35%** candidate. NOW

## 3. Post-LLM Explanations

This section now uses a **strict learning-mode LLM call** instead of the dashboard fallback helper.

That distinction matters:

- In the dashboard, fallback is good because the app should not break if Ollama fails.
- In this notebook, silent fallback is bad because you are trying to learn what the LLM changed.

So this notebook will either show real `Ollama:` output or show a clear error message. It will not pretend fallback text is post-LLM text.


In [16]:
# Learning controls
RUN_OLLAMA = True
LEARNING_LLM_MODEL = "llama3.1:latest"  # Faster for iteration. Change to "llama3.3:latest" when you want the strongest local model.
LLM_SAMPLE_LIMIT = 5  # Keep this small while learning so the notebook stays responsive.

items_for_llm = actionable[:LLM_SAMPLE_LIMIT]
llm_error = None
llm_raw_response = None
llm_payload_by_ticker = {}


def strict_generate_learning_explanations(diagnosis, items, model_name=LEARNING_LLM_MODEL):
    """Generate real LLM explanations without silently falling back.

    This intentionally differs from the dashboard helper. The dashboard must be
    resilient, so it falls back to deterministic text when Ollama fails. This
    notebook is for learning, so failure should be visible and debuggable.
    """
    payload_by_ticker = {item.ticker: _risk_action_llm_payload(diagnosis, item) for item in items}
    system_prompt = (
        "You explain portfolio risk actions in plain English. "
        "Use only the supplied JSON evidence. Do not invent news, fundamentals, prices, or future outcomes. "
        "Do not change any action label, sell amount, ranking, or confidence. "
        "Avoid jargon such as beta, market-relative risk, restrictive-rate sensitivity, 10-Q, and 10-K unless you immediately translate it. "
        "Return JSON only."
    )
    user_prompt = (
        "For each ticker, rewrite the risk action for a regular investor. "
        "Return one JSON object keyed by ticker. Each ticker value must contain exactly these keys: "
        "headline, plain_english_summary, why_now, external_context, what_improves, watchout, confidence_note. "
        "Each value should be one concise sentence. "
        "If external evidence is thin for a ticker, say that plainly.\n\n"
        f"Evidence payload by ticker:\n{json.dumps(payload_by_ticker, indent=2)}"
    )
    raw = call_ollama(
        model_name,
        [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
        temperature=0.1,
        num_predict=1800,
    )
    parsed = _extract_json_object(raw)
    cleaned = {}
    for item in items:
        value = parsed.get(item.ticker)
        if not isinstance(value, dict):
            raise ValueError(f"Ollama response did not include a JSON object for {item.ticker}. Raw response starts: {raw[:500]}")
        cleaned_item = _clean_llm_explanation(value)
        if not cleaned_item.get("headline") or not cleaned_item.get("plain_english_summary"):
            raise ValueError(f"Ollama response for {item.ticker} missed required fields: {value}")
        cleaned_item["_source"] = f"Ollama: {model_name}"
        cleaned[item.ticker] = cleaned_item
    return cleaned, raw, payload_by_ticker


if RUN_OLLAMA and items_for_llm:
    try:
        post_explanations, llm_raw_response, llm_payload_by_ticker = strict_generate_learning_explanations(
            diagnosis,
            items_for_llm,
            LEARNING_LLM_MODEL,
        )
    except Exception as exc:
        post_explanations = {}
        llm_error = repr(exc)
else:
    post_explanations = {}
    llm_error = "RUN_OLLAMA is False or there are no actionable holdings."

if llm_error:
    display(
        HTML(
            f"""
            <div style='padding:16px 18px;border:1px solid #fecaca;border-radius:16px;background:#fff1f2;color:#7f1d1d'>
                <div style='font-size:18px;font-weight:900'>No true post-LLM explanation was generated.</div>
                <div style='font-size:14px;line-height:1.55;margin-top:8px'>This is why your previous Pre/Post cards looked identical: the app helper fell back to deterministic text.</div>
                <div style='font-size:13px;line-height:1.55;margin-top:10px'><strong>Error:</strong> {llm_error}</div>
                <div style='font-size:13px;line-height:1.55;margin-top:10px'>Try using <code>LEARNING_LLM_MODEL = "llama3.1:latest"</code>, lowering <code>LLM_SAMPLE_LIMIT</code>, or confirming <code>ollama serve</code> is running.</div>
            </div>
            """
        )
    )
else:
    display(HTML(f"<div style='padding:14px 16px;border:1px solid #bae6fd;border-radius:16px;background:#f0f9ff;color:#075985'><strong>True LLM output generated.</strong> Model: {LEARNING_LLM_MODEL}. Tickers: {', '.join(post_explanations.keys())}</div>"))

post_llm_rows = [
    explanation_to_row(item, post_explanations.get(item.ticker, {}), "Post-LLM")
    for item in items_for_llm
]

post_llm_df = pd.DataFrame(post_llm_rows)
post_llm_df



,Ticker,Action,Value to sell,Reduction %,Confidence,Source,Post-LLM headline,Post-LLM summary,Post-LLM why now,Post-LLM context,Post-LLM improves,Post-LLM watchout
0,NOW,Trim 35%,$79.40,35.00%,Medium,Ollama: llama3.1:latest,Trim 35% of NOW to reduce portfolio pressure,The system recommends trimming 35% of the NOW ...,NOW has underperformed the trade-matched S&P 5...,The macro backdrop is moderately restrictive b...,Trimming the position will reduce portfolio pr...,Be cautious of company-specific risk and bench...
1,MSFT,Trim 35%,"$3,100.11",35.00%,High,Ollama: llama3.1:latest,Trim 35% of MSFT to reduce company-specific risk,The system recommends trimming 35% of the MSFT...,MSFT has underperformed the trade-matched S&P ...,The macro backdrop is moderately restrictive b...,Trimming the position will reduce concentratio...,Be cautious of recent company reports or news ...
2,SOFI,Trim 25%,$169.86,25.00%,Medium,Ollama: llama3.1:latest,Trim 25% of SOFI to reduce portfolio pressure,The system recommends trimming 25% of the SOFI...,SOFI has underperformed the trade-matched S&P ...,The macro backdrop is moderately restrictive b...,Trimming the position will reduce concentratio...,None
3,BRK.A,Trim 25%,$43.92,25.00%,Medium,Ollama: llama3.1:latest,Trim 25% of BRK.A to reduce portfolio pressure,The system recommends trimming 25% of the BRK....,BRK.A has underperformed the trade-matched S&P...,The macro backdrop is moderately restrictive b...,Trimming the position will reduce concentratio...,None
4,TCEHY,Trim 25%,$105.73,25.00%,Medium,Ollama: llama3.1:latest,Trim 25% of TCEHY to reduce portfolio pressure,The system recommends trimming 25% of the TCEH...,TCEHY has underperformed the trade-matched S&P...,The macro backdrop is moderately restrictive b...,Trimming the position will reduce concentratio...,None


## 4. Side-by-Side Comparison

This section renders one readable card per ticker, with **Pre-LLM** on the left and **Post-LLM** on the right.

Important: if the LLM failed, the right side will show a visible **No LLM output** panel. It will not duplicate the deterministic fallback and call it post-LLM.

The top strip keeps the fixed Python decisions visible: action label, amount, and confidence.



In [17]:
def html_escape(value):
    import html
    return html.escape(str(value or ""), quote=False)


def explanation_block(title, explanation, accent_color, empty_message=None):
    if not explanation:
        return f"""
        <div style="padding:16px 18px;border:1px solid #fecaca;border-radius:18px;background:#fff1f2;height:100%;">
            <div style="font-size:13px;letter-spacing:.06em;text-transform:uppercase;color:#dc2626;font-weight:900;margin-bottom:10px">{html_escape(title)}</div>
            <div style="font-size:18px;font-weight:900;line-height:1.35;margin-bottom:12px;color:#7f1d1d">No LLM output</div>
            <div style="font-size:14px;line-height:1.55;color:#7f1d1d">{html_escape(empty_message or 'The LLM call failed or was not run. Check the previous cell for the exact reason.')}</div>
        </div>
        """
    return f"""
    <div style="padding:16px 18px;border:1px solid rgba(148,163,184,.22);border-radius:18px;background:rgba(15,23,42,.04);height:100%;">
        <div style="font-size:13px;letter-spacing:.06em;text-transform:uppercase;color:{accent_color};font-weight:800;margin-bottom:10px">{html_escape(title)}</div>
        <div style="font-size:18px;font-weight:800;line-height:1.35;margin-bottom:12px">{html_escape(explanation.get('headline', ''))}</div>
        <div style="font-size:14px;line-height:1.55;margin-bottom:12px"><strong>Summary:</strong> {html_escape(explanation.get('plain_english_summary', ''))}</div>
        <div style="font-size:14px;line-height:1.55;margin-bottom:12px"><strong>Why now:</strong> {html_escape(explanation.get('why_now', ''))}</div>
        <div style="font-size:14px;line-height:1.55;margin-bottom:12px"><strong>Context:</strong> {html_escape(explanation.get('external_context', ''))}</div>
        <div style="font-size:14px;line-height:1.55;margin-bottom:12px"><strong>What improves:</strong> {html_escape(explanation.get('what_improves', ''))}</div>
        <div style="font-size:14px;line-height:1.55"><strong>Watchout:</strong> {html_escape(explanation.get('watchout', ''))}</div>
    </div>
    """


def comparison_card(item, pre, post):
    source = post.get("_source", "No LLM output") if post else "No LLM output"
    source_color = "#0ea5e9" if source.startswith("Ollama:") else "#dc2626"
    post_title = "Post-LLM explanation generated by Ollama" if source.startswith("Ollama:") else "Post-LLM explanation"
    return f"""
    <div style="border:1px solid rgba(148,163,184,.28);border-radius:22px;padding:20px;margin:0 0 24px 0;background:white;box-shadow:0 12px 32px rgba(15,23,42,.08)">
        <div style="display:flex;justify-content:space-between;gap:14px;align-items:flex-start;flex-wrap:wrap;margin-bottom:16px">
            <div>
                <div style="font-size:24px;font-weight:900">{html_escape(item.ticker)}</div>
                <div style="font-size:14px;color:#475569;margin-top:4px">Action and amount are fixed by Python. The LLM only rewrites the explanation.</div>
            </div>
            <div style="display:flex;gap:8px;flex-wrap:wrap;justify-content:flex-end">
                <span style="padding:8px 11px;border-radius:999px;background:#fee2e2;color:#991b1b;font-size:12px;font-weight:800">{html_escape(item.recommendation_label)}</span>
                <span style="padding:8px 11px;border-radius:999px;background:#dbeafe;color:#1d4ed8;font-size:12px;font-weight:800">{html_escape(money_text(item.value_to_sell))}</span>
                <span style="padding:8px 11px;border-radius:999px;background:#dcfce7;color:#166534;font-size:12px;font-weight:800">{html_escape(item.confidence_band)} confidence</span>
                <span style="padding:8px 11px;border-radius:999px;background:#e0f2fe;color:{source_color};font-size:12px;font-weight:800">Post source: {html_escape(source)}</span>
            </div>
        </div>
        <div style="display:grid;grid-template-columns:repeat(2,minmax(300px,1fr));gap:18px;align-items:stretch">
            {explanation_block('Pre-LLM deterministic explanation', pre, '#2563eb')}
            {explanation_block(post_title, post, '#0ea5e9', empty_message='No true LLM rewrite exists for this ticker yet. The previous cell shows the Ollama error or run setting.')}
        </div>
    </div>
    """

cards = []
for item in items_for_llm:
    pre = _risk_action_deterministic_explanation(item)
    post = post_explanations.get(item.ticker)
    cards.append(comparison_card(item, pre, post))

if cards:
    display(HTML("".join(cards)))
else:
    display(HTML("<div style='padding:16px;border:1px solid #cbd5e1;border-radius:14px'>No actionable holdings were available in the latest diagnosis.</div>"))



## 5. Audit Checklist

Use this checklist when reviewing LLM output:

- Did the **action label** stay the same?
- Did the **sell amount** stay the same?
- Did the LLM explain **why now** in regular language?
- Did the LLM mention external evidence only when evidence exists?
- Did the LLM keep uncertainty visible?
- Would a non-technical investor understand the recommendation in 10 seconds?

If the answer is no, improve the prompt or the evidence payload. Do not let the LLM override the deterministic object.

In [18]:
audit_rows = []
for item in items_for_llm:
    post = post_explanations.get(item.ticker, {})
    source = post.get("_source", "No LLM output")
    audit_rows.append(
        {
            "Ticker": item.ticker,
            "Action fixed by Python": item.recommendation_label,
            "Amount fixed by Python": money_text(item.value_to_sell),
            "Explanation source": source,
            "LLM actually used?": source.startswith("Ollama:"),
            "Has why-now sentence?": bool(post.get("why_now")),
            "Has watchout?": bool(post.get("watchout")),
            "Has confidence note?": bool(post.get("confidence_note")),
        }
    )

pd.DataFrame(audit_rows)



,Ticker,Action fixed by Python,Amount fixed by Python,Explanation source,LLM actually used?,Has why-now sentence?,Has watchout?,Has confidence note?
0,NOW,Trim 35%,$79.40,Ollama: llama3.1:latest,True,True,True,True
1,MSFT,Trim 35%,"$3,100.11",Ollama: llama3.1:latest,True,True,True,True
2,SOFI,Trim 25%,$169.86,Ollama: llama3.1:latest,True,True,True,True
3,BRK.A,Trim 25%,$43.92,Ollama: llama3.1:latest,True,True,True,True
4,TCEHY,Trim 25%,$105.73,Ollama: llama3.1:latest,True,True,True,True


## What You Should Learn From This Notebook

This is the pattern we will reuse across the project:

1. Build a deterministic object first.
2. Create a compact evidence payload from that object.
3. Ask the LLM to explain, not decide.
4. Label the output source clearly.
5. Keep fallback behavior if the LLM is unavailable.
6. Compare pre-LLM and post-LLM output before trusting it in the dashboard.

That is the core agentic lifecycle in this app.